# Phase 10: Hybrid Wave+Byte Universal Generation

**The Best of Both Worlds — Precision When Needed, Speed When Possible**

This notebook implements the Phase 10 hybrid generation system:
- **Byte-mode** (Phase 8 WaveDecoder): Precise, character-perfect (~100 steps/sentence)
- **Wave-mode** (Phase 10 modules): Fast, semantic (~15 steps/sentence → 6x faster)
- **TaskRouter**: Intelligent automatic mode selection

## Key Innovation
```
Input → CSE → Field Query → TaskRouter
                              │
              ┌───────────────┴───────────────┐
              ▼                               ▼
         FAST MODE                      PRECISE MODE
       (WaveGenerator)                  (ByteDecoder)
              │                               │
       WaveToText                             │
              │                               │
              └───────────────┬───────────────┘
                              ▼
                        Final Output
```

## Loading Strategy
- **Flux-X-complete.flx**: Full trained base model (all components: CSE, Field, GR, TL, Memory, Decoder, bridges)
- **Flux-capable.flx**: Enriched field from Phase 9.7 (155k+ samples for best text generation)

Phase 10 loads **Flux-X-complete.flx** as the foundation, then **merges the enriched field** from Flux-capable.flx for optimal generation quality.

---

## Cell 1: Clone/Pull FLUX Repository

In [ ]:
%%time
import os
from pathlib import Path

REPO_URL = 'https://github.com/Unseengap/FLUX.git'
ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')

if ROOT.exists():
    print('  Pulling latest...')
    !cd {ROOT} && git pull
else:
    print('  Cloning repo...')
    !git clone {REPO_URL} {ROOT}

os.chdir(ROOT)
print(f'  Working dir: {os.getcwd()}')

# Install dependencies
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
!pip install -q huggingface_hub datasets transformers

print('  ✓ Dependencies installed')

## Cell 2: Setup Paths & Logger

In [ ]:
import sys
from pathlib import Path

ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')
ROOT_DIR = ROOT
PHASES_DIR = ROOT / 'phases'
CHECKPOINT_DIR = ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Add all phase paths
for i in range(1, 11):
    p = PHASES_DIR / f'phase{i}'
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Add phase subdirs
for subdir in ['phase8_8', 'phase8_9', 'phase10']:
    p = PHASES_DIR / subdir
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        print(f'  ✓ {subdir} found')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from flux_utils import PhaseLogger, get_device

log = PhaseLogger(phase=10)
log.separator("Phase 10: Hybrid Wave+Byte Universal Generation")

print('  ✓ Paths configured')

## Cell 3: Hardware & Secrets

In [ ]:
import torch
import os

log.cell_start("Cell 3 — Hardware & Secrets")

device = get_device()
print(f'  Device: {device}')

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU: {gpu_name} ({vram:.1f} GB)')

# Load secrets (Kaggle → Colab → env fallback)
hf_token = None

# Try Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    print('  ✓ Kaggle secrets loaded')
except:
    pass

# Try Colab secrets
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        print('  ✓ Colab secrets loaded')
    except:
        pass

# Fall back to env vars
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        print('  ✓ Environment variables loaded')

# Clean and set token
if hf_token:
    hf_token = hf_token.strip()
    os.environ['HF_TOKEN'] = hf_token
    print('  ✓ HF_TOKEN set')
else:
    print('  ⚠ HF_TOKEN not found (will try anonymous download)')

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

## Cell 4: Download & Load FLUXHybrid

**Loading Strategy:**
1. **Flux-X-complete.flx**: Full trained base model (all components)
2. **Flux-capable.flx**: Enriched field (155k+ samples, best text generation)

The hybrid model uses X-complete for all trained components (CSE, Field, GR, TL, Memory, Decoder, bridges) and merges the capable field for optimal generation quality.

In [ ]:
log.cell_start("Cell 4 — Download & Load FLUXHybrid")

from huggingface_hub import hf_hub_download
from pathlib import Path

print("  Phase 10 Loading Strategy:")
print("  ├── Flux-X-complete.flx: Full trained base model")
print("  └── Flux-capable.flx: Enriched field (best generation)")
print()

# Step 1: Download Flux-X-complete.flx (base model with all trained components)
BASE_FLX = 'Flux-X-complete.flx'
base_path = CHECKPOINT_DIR / BASE_FLX

if base_path.exists():
    size_mb = base_path.stat().st_size / 1e6
    print(f'  ✓ Found local {BASE_FLX} ({size_mb:.1f} MB)')
else:
    print(f'  Downloading {BASE_FLX} (full trained base model)...')
    try:
        downloaded = hf_hub_download(
            repo_id='UnseenGAP/FLUX',
            filename=f'checkpoints/{BASE_FLX}',
            local_dir=str(ROOT_DIR),
            token=hf_token,
        )
        size_mb = Path(downloaded).stat().st_size / 1e6
        print(f'  ✓ Downloaded {BASE_FLX} ({size_mb:.1f} MB)')
    except Exception as e:
        print(f'  ⚠ Base model download failed: {e}')
        # Fallback to Flux-beta.flx
        BASE_FLX = 'Flux-beta.flx'
        base_path = CHECKPOINT_DIR / BASE_FLX
        if not base_path.exists():
            hf_hub_download(
                repo_id='UnseenGAP/FLUX',
                filename=f'checkpoints/{BASE_FLX}',
                local_dir=str(ROOT_DIR),
                token=hf_token,
            )
        print(f'  ✓ Using fallback: {BASE_FLX}')

# Step 2: Download Flux-capable.flx (enriched field for best generation)
CAPABLE_FLX = 'Flux-capable.flx'
capable_path = CHECKPOINT_DIR / CAPABLE_FLX

if capable_path.exists():
    size_mb = capable_path.stat().st_size / 1e6
    print(f'  ✓ Found local {CAPABLE_FLX} ({size_mb:.1f} MB)')
else:
    print(f'  Downloading {CAPABLE_FLX} (enriched field)...')
    try:
        downloaded = hf_hub_download(
            repo_id='UnseenGAP/FLUX',
            filename=f'checkpoints/{CAPABLE_FLX}',
            local_dir=str(ROOT_DIR),
            token=hf_token,
        )
        size_mb = Path(downloaded).stat().st_size / 1e6
        print(f'  ✓ Downloaded {CAPABLE_FLX} ({size_mb:.1f} MB)')
    except Exception as e:
        print(f'  ⚠ Capable field download failed: {e}')
        print(f'    (Will use base field only)')

# Step 3: Load FLUXHybrid with merged capable field
print(f'\n  Loading FLUXHybrid...')
print(f'    Base: {BASE_FLX}')
print(f'    Field: {CAPABLE_FLX if capable_path.exists() else "using base"}')

from flux_hybrid import FLUXHybrid

model = FLUXHybrid.from_flx(
    path=str(base_path),
    device=device,
    verbose=True,
    init_wave_modules=True,
    merge_capable_field=capable_path.exists(),  # Merge if available
)

# Show stats
stats = model.get_stats()
print(f'\n  Model Statistics:')
print(f'    Total params: {stats["total_params"]:,}')
print(f'    Version: {stats["version"]}')
print(f'    Wave mode available: {stats["wave_mode_available"]}')
print(f'    Field shape: {stats["field_shape"]}')
print(f'    Field energy: {stats["field_energy"]:.4f}')
print(f'    Field mass sum: {stats["field_mass_sum"]:.2f}')

log.metric("Base model", BASE_FLX)
log.metric("Total params", f"{stats['total_params']:,}")
log.metric("Wave mode", str(stats['wave_mode_available']))

log.cell_end("Cell 4 — Download & Load FLUXHybrid", "PASS")

## Cell 5: TaskRouter Demo

The TaskRouter automatically decides between:
- **Wave mode** (fast): For chat, explanations, summaries
- **Byte mode** (precise): For code, URLs, exact quotes

Let's see it in action!

In [ ]:
log.cell_start("Cell 5 — TaskRouter Demo")

print("\n  TaskRouter: Intelligent Mode Selection")
print("  " + "="*60)

# Demo prompts with expected routing
demo_prompts = [
    # Wave-expected (semantic/conversational)
    ("Explain artificial intelligence", "Semantic explanation"),
    ("What is the meaning of life?", "Philosophical question"),
    ("Tell me about the solar system", "Educational content"),
    ("Summarize machine learning", "Summary request"),
    
    # Byte-expected (precision needed)
    ("```python\ndef hello():\n    print('hi')\n```", "Code block"),
    ("def factorial(n):", "Python function"),
    ("https://github.com/Unseengap/FLUX", "URL"),
    ("Write a function to check if prime", "Code request"),
]

print("\n  Routing Decisions:")
print("  " + "-"*60)

for prompt, description in demo_prompts:
    mode, reason = model.task_router.route_with_reason(prompt)
    mode_icon = "🌊" if mode == 'wave' else "🔢"
    
    display_prompt = prompt[:35] + "..." if len(prompt) > 35 else prompt
    print(f"\n  {mode_icon} [{description}]")
    print(f"     Prompt: \"{display_prompt}\"")
    print(f"     Mode: {mode.upper()}")
    print(f"     Reason: {reason}")

# Show stats
router_stats = model.task_router.get_stats()
print("\n  " + "-"*60)
print(f"  Routing Statistics:")
print(f"    Total decisions: {router_stats['total_routes']}")
print(f"    Wave mode: {router_stats['route_counts']['wave']} ({100*router_stats['wave_ratio']:.0f}%)")
print(f"    Byte mode: {router_stats['route_counts']['byte']} ({100*router_stats['byte_ratio']:.0f}%)")

log.cell_end("Cell 5 — TaskRouter Demo", "PASS")

## Cell 6: Wave vs Byte Comparison

Side-by-side comparison of both generation modes on the same prompts.

In [ ]:
log.cell_start("Cell 6 — Wave vs Byte Comparison")

import time

print("\n  Side-by-Side Generation Comparison")
print("  " + "="*60)

test_prompts = [
    "The capital of France is",
    "Explain quantum computing:",
    "def fibonacci(n):",
]

wave_times = []
byte_times = []

for prompt in test_prompts:
    print(f"\n  📝 Prompt: \"{prompt}\"")
    print("  " + "-"*50)
    
    # Wave mode
    print("\n  🌊 WAVE MODE (fast, semantic)")
    start = time.time()
    wave_response = model.generate(prompt, max_length=50, mode='wave', temperature=0.7)
    wave_time = time.time() - start
    wave_times.append(wave_time)
    
    print(f"     Time: {wave_time:.3f}s")
    print(f"     Steps: {wave_response.n_steps}")
    if wave_response.text:
        display_text = wave_response.text.replace('\n', ' ')[:60]
        print(f"     Output: \"{display_text}...\"")
    
    # Byte mode
    print("\n  🔢 BYTE MODE (precise)")
    start = time.time()
    byte_response = model.generate(prompt, max_length=50, mode='byte', temperature=0.7)
    byte_time = time.time() - start
    byte_times.append(byte_time)
    
    print(f"     Time: {byte_time:.3f}s")
    if byte_response.text:
        display_text = byte_response.text.replace('\n', ' ')[:60]
        print(f"     Output: \"{display_text}...\"")

# Overall stats
print("\n  " + "="*60)
print("  OVERALL STATISTICS")
print("  " + "="*60)

avg_wave = sum(wave_times) / max(1, len(wave_times))
avg_byte = sum(byte_times) / max(1, len(byte_times))

print(f"\n  Average generation time:")
print(f"    Wave mode: {avg_wave:.3f}s")
print(f"    Byte mode: {avg_byte:.3f}s")

if avg_wave > 0:
    speedup = avg_byte / avg_wave
    print(f"    Speedup: {speedup:.1f}x {'(wave faster)' if speedup > 1 else '(byte faster)'}")

log.metric("Wave avg time", f"{avg_wave:.3f}s")
log.metric("Byte avg time", f"{avg_byte:.3f}s")

log.cell_end("Cell 6 — Wave vs Byte Comparison", "PASS")

## Cell 7: Automatic Hybrid Generation

Let's test the full hybrid pipeline with auto-routing.

In [ ]:
log.cell_start("Cell 7 — Automatic Hybrid Generation")

print("\n  Automatic Hybrid Generation (mode='auto')")
print("  " + "="*60)
print("  The TaskRouter decides the best mode for each prompt.")

auto_prompts = [
    "Hello! How are you today?",
    "Explain machine learning in simple terms",
    "```python\nfor i in range(10):\n    print(i)\n```",
    "What is the capital of Japan?",
    "Write a function to calculate factorial",
    "Tell me a short story about a robot",
]

print("\n  Auto-Routed Generations:")
print("  " + "-"*60)

for prompt in auto_prompts:
    response = model.generate(
        prompt,
        max_length=60,
        mode='auto',
        temperature=0.8,
    )
    
    mode_icon = "🌊" if response.mode == 'wave' else "🔢"
    
    display_prompt = prompt[:40] + "..." if len(prompt) > 40 else prompt
    print(f"\n  {mode_icon} \"{display_prompt}\"")
    print(f"     Mode: {response.mode} | Time: {response.generation_time:.3f}s")
    print(f"     Reason: {response.mode_reason}")
    
    if response.text:
        display_text = response.text.replace('\n', ' ')[:70]
        print(f"     Output: \"{display_text}\"")

log.cell_end("Cell 7 — Automatic Hybrid Generation", "PASS")

## Cell 8: Run Phase 10 Tests

In [ ]:
log.cell_start("Cell 8 — Run Phase 10 Tests")

import subprocess
import sys

tests = [
    'test_phase10_test1.py',
    'test_phase10_test2.py', 
    'test_phase10_test3.py',
]

phase10_dir = PHASES_DIR / 'phase10'
test_results = {}

print("\n  Running Phase 10 Tests")
print("  " + "="*60)

for test in tests:
    test_path = phase10_dir / test
    print(f"\n  Running: {test}")
    
    if test_path.exists():
        try:
            result = subprocess.run(
                [sys.executable, str(test_path)],
                capture_output=True,
                text=True,
                timeout=120,
                cwd=str(phase10_dir),
            )
            
            passed = result.returncode == 0
            test_results[test] = passed
            
            status = "✓ PASS" if passed else "✗ FAIL"
            print(f"    {status}")
            
            # Show first few lines of output
            if result.stdout:
                lines = result.stdout.strip().split('\n')
                for line in lines[:5]:
                    if line.strip():
                        print(f"      {line[:70]}")
            
            if not passed and result.stderr:
                print(f"    Error: {result.stderr[:100]}")
                
        except subprocess.TimeoutExpired:
            print(f"    ⚠ TIMEOUT")
            test_results[test] = False
        except Exception as e:
            print(f"    ✗ ERROR: {e}")
            test_results[test] = False
    else:
        print(f"    ⚠ Test file not found")
        test_results[test] = None

# Summary
print("\n  " + "="*60)
print("  TEST SUMMARY")
print("  " + "="*60)

passed = sum(1 for v in test_results.values() if v is True)
total = len(tests)

for test, result in test_results.items():
    if result is True:
        print(f"    ✓ {test}")
    elif result is False:
        print(f"    ✗ {test}")
    else:
        print(f"    ? {test} (not found)")

print(f"\n    Result: {passed}/{total} tests passed")

log.cell_end("Cell 8 — Run Phase 10 Tests", "PASS" if passed == total else "PARTIAL")

## Cell 9: Phase 10 Summary

In [ ]:
log.separator("Phase 10 Complete: Hybrid Wave+Byte Generation")

print("""
╔════════════════════════════════════════════════════════════════════╗
║                    PHASE 10 COMPLETE                              ║
║          HYBRID WAVE+BYTE UNIVERSAL GENERATION                    ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  LOADING STRATEGY:                                                 ║
║  ├── Flux-X-complete.flx: Full trained base (all components)     ║
║  └── Flux-capable.flx: Enriched field (155k+ samples injected)   ║
║                                                                    ║
║  GENERATION MODES:                                                 ║
║  ├── 🌊 WAVE MODE (fast, semantic)                                ║
║  │   └── ~6x faster than byte mode                                ║
║  │   └── Good for: chat, explanations, summaries                  ║
║  │                                                                 ║
║  └── 🔢 BYTE MODE (precise, character-perfect)                    ║
║      └── Same as Phase 8 WaveDecoder                              ║
║      └── Good for: code, URLs, exact quotes                       ║
║                                                                    ║
║  ROUTING:                                                          ║
║  ├── TaskRouter analyzes prompts                                  ║
║  ├── Auto-selects best mode                                       ║
║  └── Pattern matching + configurable rules                        ║
║                                                                    ║
║  MULTIMODAL EXTENSION (Phase 8.8 foundation):                     ║
║  ├── Same waves → different decoders                              ║
║  ├── WaveToText: UTF-8 text                                       ║
║  ├── WaveToImage: RGB images (3 physics engines)                  ║
║  └── WaveToAudio, WaveToMol: Future extensions                    ║
║                                                                    ║
║  THIS IS FLUX:                                                     ║
║  ├── One wave representation, multiple outputs                    ║
║  ├── Physics-based learning (no gradients for injection)          ║
║  └── Hybrid generation for best of both worlds                    ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
""")

# Final stats
final_stats = model.get_stats()
router_stats = model.task_router.get_stats()

print("\nFINAL STATISTICS:")
print(f"  Base model: {BASE_FLX}")
print(f"  Capable field: {'merged' if capable_path.exists() else 'not available'}")
print(f"  Total params: {final_stats['total_params']:,}")
print(f"  Wave mode available: {final_stats['wave_mode_available']}")
print(f"  Field shape: {final_stats['field_shape']}")
print(f"  Field mass sum: {final_stats['field_mass_sum']:.2f}")
print()
print(f"  Router decisions:")
print(f"    Wave mode: {router_stats['route_counts']['wave']}")
print(f"    Byte mode: {router_stats['route_counts']['byte']}")

log.success("Phase 10 complete!")